# DATA209 — Advanced Exploratory Data Analysis
# Practical P3-4 · Type handling, text and time series

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 2 · Module 1 · CO1

---

**Objective.** Audit and correct data types, measure word frequency in free text, plot a time series, and quantify categorical imbalance.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.

This session also uses `Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv` for the text exercise and `HistoricalPrices.csv` for the time series. Both fall back gracefully if absent.

### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 — the dataset and the dependent/independent split.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P3-4 — Type handling, text and time series

### Check datatypes

The dtype pandas guessed is not necessarily the *measurement scale* of the variable.
`OperatingSystems`, `Browser`, `Region` and `TrafficType` are stored as integers but are
**nominal codes** — averaging them is meaningless. This is the single most common error in
submitted assignments.

In [ ]:
# ---- Audit what pandas guessed -----------------------------------------
audit = pd.DataFrame({
    "pandas_dtype": df.dtypes.astype(str),
    "unique"      : df.nunique(),
    "sample"      : [df[c].dropna().unique()[:4] for c in df.columns],
})

# a low unique count on an integer column is a strong hint that it is a code, not a quantity
audit["suspect_nominal"] = (
    df.dtypes.isin([np.dtype("int64")]) & (df.nunique() <= 25)
)
print(audit.to_string())

print("\nColumns stored as integers but almost certainly nominal codes:")
print(" ", audit[audit["suspect_nominal"]].index.tolist())

In [ ]:
# ---- Convert types -----------------------------------------------------
dfc = df.copy()

# 1 — nominal codes: integer -> category (blocks accidental arithmetic)
nominal_codes = ["OperatingSystems", "Browser", "Region", "TrafficType"]
for c in nominal_codes:
    dfc[c] = dfc[c].astype("category")

# 2 — Month is ordinal: give it a real order so it sorts and plots correctly
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "June",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
present = [m for m in month_order if m in dfc["Month"].unique()]
dfc["Month"] = pd.Categorical(dfc["Month"], categories=present, ordered=True)

# 3 — text categories: strip and standardise before anything counts them
dfc["VisitorType"] = dfc["VisitorType"].astype(str).str.strip()

# 4 — booleans stay boolean
for c in ["Weekend", "Revenue"]:
    dfc[c] = dfc[c].astype(bool)

print("After conversion:")
print(dfc.dtypes.astype(str).to_string())
print("\nMonth is now ordered:", list(dfc["Month"].cat.categories))
print("Months absent from the data:", [m for m in month_order if m not in present])

In [ ]:
# ---- Why the ordered category matters ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))

# alphabetical (wrong) vs chronological (right)
df["Month"].value_counts().plot(kind="bar", ax=axes[0], color="#8B9199")
axes[0].set_title("Unordered — pandas sorts by frequency or alphabetically")

dfc["Month"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#3B6E8F")
axes[1].set_title("Ordered Categorical — calendar order, trend readable")

plt.tight_layout(); plt.show()
print("The left chart makes seasonality invisible. Same data, different dtype.")

### Text word frequency

The shoppers dataset has no free text, so this section uses the **Amazon consumer reviews**
corpus from the course Drive. If the file is absent the cell falls back to a small built-in
corpus so the notebook still runs.

In [ ]:
# ---- Load a text column ------------------------------------------------
text_path = find("Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv")

if text_path:
    reviews = pd.read_csv(text_path, usecols=["reviews.text"], low_memory=False)
    corpus  = reviews["reviews.text"].dropna().astype(str)
    print(f"Loaded {len(corpus):,} reviews from {os.path.basename(text_path)}")
else:
    corpus = pd.Series([
        "the website is easy to use and checkout was fast",
        "great discounts on products but delivery was slow",
        "customer service did not respond to my query",
        "excellent product quality and fast delivery",
        "the checkout page kept failing on mobile",
    ])
    print("Amazon file not found — using a 5-document fallback corpus.")

print("\nDocument length (characters):")
print(corpus.str.len().describe().round(1).to_string())

In [ ]:
# ---- Word frequency ----------------------------------------------------
import re
from collections import Counter

STOP = set("""a an the and or but if of to in on for with at by from is are was were be been
being it its this that these those i you he she they we as not no so than then there here
have has had do does did will would can could my your our their me him her them what which
who when where why how all any both each more most other some such only own same too very
s t don now just""".split())

def tokenise(s):
    return [w for w in re.findall(r"[a-z']+", s.lower()) if len(w) > 2 and w not in STOP]

sample  = corpus.sample(min(5000, len(corpus)), random_state=RANDOM_STATE)
counts  = Counter()
for doc in sample:
    counts.update(tokenise(doc))

freq = (pd.Series(counts).sort_values(ascending=False).head(20)
          .rename("count").to_frame())
freq["share_%"] = (freq["count"] / sum(counts.values()) * 100).round(2)
print(f"Vocabulary size: {len(counts):,} distinct tokens")
print(freq.to_string())

plt.figure(figsize=(9, 4))
sns.barplot(x=freq["count"], y=freq.index, color="#3B6E8F")
plt.title("Top 20 tokens (stop words removed)"); plt.xlabel("count"); plt.ylabel("")
plt.tight_layout(); plt.show()

print("\nA ranked frequency bar is precise. A word cloud is decorative — it cannot be read off.")

### Time-series plotting

`HistoricalPrices.csv` holds a daily price series. Time-ordered data has a rule the rest of the
course depends on: **you may not shuffle the rows, and you may not split them randomly.**

In [ ]:
# ---- Time series -------------------------------------------------------
ts_path = find("HistoricalPrices.csv")

if ts_path:
    ts = pd.read_csv(ts_path)
    ts.columns = [c.strip() for c in ts.columns]      # this file has padded headers
    ts["Date"] = pd.to_datetime(ts["Date"], errors="coerce")
    ts = ts.dropna(subset=["Date"]).sort_values("Date").set_index("Date")
    series = ts["Close"]
    label  = "Daily close"
else:
    idx = pd.date_range("2020-01-01", periods=900, freq="D")
    series = pd.Series(np.cumsum(np.random.randn(900)) + 100, index=idx)
    label  = "Synthetic series (HistoricalPrices.csv not found)"

roll30, roll90 = series.rolling(30).mean(), series.rolling(90).mean()

plt.figure(figsize=(11, 3.6))
plt.plot(series.index, series.values, lw=0.8, color="#8B9199", label=label)
plt.plot(roll30.index, roll30.values, lw=1.8, color="#3B6E8F", label="30-day mean")
plt.plot(roll90.index, roll90.values, lw=1.8, color="#B5432E", label="90-day mean")
plt.legend(); plt.title("Level and trend"); plt.tight_layout(); plt.show()

print("Span:", series.index.min().date(), "to", series.index.max().date(),
      f"| {len(series):,} observations")
print("Rolling means separate long-run trend from day-to-day noise without any modelling.")

In [ ]:
# ---- Seasonality in the shoppers data ----------------------------------
monthly = (dfc.groupby("Month", observed=True)
              .agg(sessions=("Revenue", "size"),
                   conversion=("Revenue", "mean")))
monthly["conversion"] *= 100

fig, ax1 = plt.subplots(figsize=(10, 3.4))
ax1.bar(monthly.index.astype(str), monthly["sessions"], color="#C9D4DB", label="sessions")
ax1.set_ylabel("sessions"); ax1.set_xlabel("")
ax2 = ax1.twinx()
ax2.plot(monthly.index.astype(str), monthly["conversion"], color="#B5432E",
         marker="o", lw=2, label="conversion %")
ax2.set_ylabel("conversion %")
ax1.set_title("Volume and conversion by month")
plt.tight_layout(); plt.show()

print(monthly.round(2).to_string())
print("\nNote the dual axis. It is used here deliberately and labelled — but a dual axis can")
print("manufacture an apparent relationship by choosing the two ranges. Use it sparingly.")

### Categorical distribution and imbalance

Imbalance is a univariate finding with consequences that reach all the way to model evaluation.
Record it now.

In [ ]:
# ---- Categorical imbalance --------------------------------------------
cat_cols = ["VisitorType", "Weekend", "Month", "OperatingSystems",
            "Browser", "Region", "TrafficType"]

rows = []
for c in cat_cols:
    vc = dfc[c].value_counts(normalize=True, dropna=False)
    rows.append({
        "column"        : c,
        "levels"        : dfc[c].nunique(),
        "top_level"     : str(vc.index[0]),
        "top_share_%"   : round(vc.iloc[0] * 100, 1),
        "levels_under_1%": int((vc < 0.01).sum()),
    })
imbalance = pd.DataFrame(rows).sort_values("top_share_%", ascending=False)
print(imbalance.to_string(index=False))

print("\nTarget:", f"{y.mean()*100:.1f}% positive")
print("\nInterpretation")
print("- top_share_% above ~90 means the column is nearly constant and carries little signal.")
print("- levels_under_1% counts rare categories: group these into 'Other' before encoding (P21-22).")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, c in zip(axes, ["VisitorType", "Weekend", "Region"]):
    (dfc[c].value_counts(normalize=True) * 100).head(10).plot(
        kind="bar", ax=ax, color="#3B6E8F")
    ax.set_title(f"{c} (% of sessions)"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### Deliverable — P3-4

A notebook plus a short table listing **every type correction you made and why**, together with
the word-frequency plot, the time-series plot and the imbalance table.